# Pipeline de preprocessing — Adult Income (UCI)

**Objectif :** montrer concrètement les **effets de la pipeline de feature
engineering** (`adult_income.preprocessing.PreprocessingPipeline`) sur le jeu
de données, en appliquant les décisions du **bilan §9 de l'EDA** : un traitement
adapté **par feature** plutôt qu'une recette uniforme.

La pipeline est un `ColumnTransformer` scikit-learn composé de 7 branches :

| Branche | Colonnes | Traitement |
|---|---|---|
| `num` | numériques « normales » (age, education-num, hours-per-week) | médiane + `StandardScaler` |
| `num_log` | `capital-gain` | médiane + `log1p` + `RobustScaler` (queue lourde conservée) |
| `num_bin` | `capital-gain`, `capital-loss` | indicateur binaire `has_*` (décomposition « hurdle ») |
| `cat` | catégorielles faible cardinalité (marital-status, relationship, race, sex) | `?`→NaN + mode + `OneHotEncoder` |
| `cat_unknown` | `workclass` | `?`→NaN + catégorie **`Unknown`** + `OneHotEncoder` |
| `cat_occupation` | `occupation` | `?`→NaN + catégorie **`Unknown`** + `TargetEncoder` |
| `cat_country` | `native-country` | `?`→NaN + mode + regroupement des modalités `<50` + `OneHotEncoder` |

Points clés du bilan §9 :
- **`capital-loss`** n'est plus log-transformée : un simple indicateur binaire suffit (magnitude peu informative). **`capital-gain`** garde le `log1p` **et** reçoit un indicateur binaire.
- **`workclass` / `occupation`** : la manquance est **informative** (~9,5 % `>50K` contre 24 % en moyenne), on la conserve via une catégorie `Unknown` au lieu de l'imputer par le mode.
- **`native-country`** : manquance non informative → imputation par le mode (US) ; ses ~20 modalités rares (`<50` obs) sont regroupées.
- **`education` / `fnlwgt`** : supprimées (`remainder="drop"`).

> ⚠️ La pipeline est **ajustée sur le train uniquement**, puis
> appliquée au test. C'est essentiel ici car le `TargetEncoder` utilise la cible
> `y`, l'ajuster sur le test provoquerait une fuite de données (*data leakage*).


## 0. Préparation de l'environnement

In [ ]:
# Recharge automatique des modules src/ modifiés sans restart du kernel
%load_ext autoreload
%autoreload 2

# Permet d'importer le package src/ depuis le dossier notebooks/
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from adult_income import (
    DataLoader,
    PreprocessingPipeline,
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET,
    POSITIVE_CLASS,
    NEGATIVE_CLASS,
)
from adult_income.data.schemas import (
    NORMAL_NUMERICAL_FEATURES,
    LOG_NUMERICAL_FEATURES,
    BINARY_NUMERICAL_FEATURES,
    MODE_CATEGORICAL_FEATURES,
    UNKNOWN_ONEHOT_FEATURES,
    UNKNOWN_TARGET_FEATURES,
    GROUPED_CATEGORICAL_FEATURES,
    RARE_CATEGORY_MIN_FREQUENCY,
    FEATURES_TO_DROP,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Imports OK")


## 1. Chargement et découpage train / test

In [ ]:
loader = DataLoader(test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = loader.load_and_split()

print(f"Train : {X_train.shape[0]:,} obs  -  Test : {X_test.shape[0]:,} obs")
print(f"Features brutes : {X_train.shape[1]}")
X_train.head()


On ajuste **tout sur le train**. La cible textuelle (`>50K` / `<=50K`) est
encodée en 0/1 via le `LabelEncoder` interne de la pipeline.

In [ ]:
pipe = PreprocessingPipeline(encoding="onehot", scaling=True)

y_train_enc = pipe.fit_transform_target(y_train)
y_test_enc = pipe.transform_target(y_test)

print("Classes (ordre LabelEncoder) :", list(pipe.classes_))
print("Répartition train encodée   :", np.bincount(y_train_enc))


## 2. La pipeline de preprocessing — vue d'ensemble

`build()` retourne un `ColumnTransformer` prêt à être ajusté. On affiche son
diagramme : chaque branche cible un sous-ensemble de colonnes (les features
listées dans `FEATURES_TO_DROP` sont ignorées via `remainder="drop"`).

In [ ]:
print("Numériques normales (StandardScaler) :", NORMAL_NUMERICAL_FEATURES)
print("Numériques log1p (RobustScaler)       :", LOG_NUMERICAL_FEATURES)
print("Indicateurs binaires has_*            :", BINARY_NUMERICAL_FEATURES)
print("Catégorielles one-hot (mode)          :", MODE_CATEGORICAL_FEATURES)
print("Catégorielles one-hot + 'Unknown'     :", UNKNOWN_ONEHOT_FEATURES)
print("Catégorielles target-encoding         :", UNKNOWN_TARGET_FEATURES)
print(f"Catégorielles regroupées (<{RARE_CATEGORY_MIN_FREQUENCY})        :", GROUPED_CATEGORICAL_FEATURES)
print("Features ignorées (drop)              :", FEATURES_TO_DROP)

pre = pipe.build()
pre


## 3. Application de la pipeline (`fit_transform` sur le train)

In [ ]:
X_train_t = pre.fit_transform(X_train, y_train_enc)
feat_names = pre.get_feature_names_out()
X_train_t = pd.DataFrame(X_train_t, columns=feat_names, index=X_train.index)

print(f"Avant : {X_train.shape[1]} colonnes  ->  Apres : {X_train_t.shape[1]} colonnes")
X_train_t.head()


Le `OneHotEncoder` fait exploser la dimensionnalité (une colonne par modalité),
tandis que le `TargetEncoder` garde une seule colonne par variable. Toutes les
sorties sont désormais **numériques**, prêtes pour un estimateur.

## 4. Effets détaillés de la pipeline

### 4.1 Valeurs manquantes & `?`

Les `?` du dataset Adult sont convertis en `NaN` **dès le chargement** (`DataLoader`).
La pipeline garantit qu'il ne reste **aucun manquant** en sortie, mais avec une
stratégie **par variable** plutôt qu'une imputation uniforme :

- **`workclass` / `occupation`** → catégorie **`Unknown`** (manquance informative, conservée) ;
- **`native-country`** → imputation par le **mode** (US, manquance non informative).

> Un `replace_qm` reste présent dans chaque branche catégorielle **par sécurité**
> (idempotent) : si la pipeline reçoit des données brutes contenant encore des `?`,
> ils sont reconvertis en `NaN` avant imputation.


In [ ]:
# Les ? sont deja convertis en NaN au chargement -> on compte les NaN
raw_missing = X_train[CATEGORICAL_FEATURES].isna().sum()
print("Valeurs manquantes (NaN) dans le train brut :")
print(raw_missing[raw_missing > 0].to_string())

n_nan = int(np.isnan(X_train_t.values).sum())
print(f"\nNaN dans la matrice transformee : {n_nan}")


### 4.2 Mise à l'échelle des variables numériques

Le `StandardScaler` recentre (moyenne ≈ 0) et réduit (écart-type ≈ 1) les
variables numériques normales — indispensable pour les modèles sensibles à
l'échelle (régression logistique, SVM, KNN).

In [ ]:
fig, axes = plt.subplots(2, len(NORMAL_NUMERICAL_FEATURES), figsize=(16, 7))
for j, col in enumerate(NORMAL_NUMERICAL_FEATURES):
    sns.histplot(X_train[col], ax=axes[0, j], bins=40, color="steelblue")
    axes[0, j].set_title(f"{col} - brut")
    sns.histplot(X_train_t[f"num__{col}"], ax=axes[1, j], bins=40, color="darkorange")
    axes[1, j].set_title(f"{col} - StandardScaler")
fig.suptitle("Effet du StandardScaler (variables numeriques normales)", fontsize=14)
fig.tight_layout()
plt.show()

stats = X_train_t[[f"num__{c}" for c in NORMAL_NUMERICAL_FEATURES]].agg(["mean", "std"]).round(3)
print(stats.to_string())


### 4.3 Variables `capital-*` : log1p + indicateurs binaires

`capital-gain` et `capital-loss` sont **zero-inflated** (>90 % de zéros) et très
asymétriques. On les décompose façon « hurdle » :

- **`capital-gain`** → `log1p` + `RobustScaler` : on comprime la queue lourde
  tout en **conservant la magnitude** (un gros gain est très prédictif de `>50K`).
- **`capital-loss`** → **pas** de log : sa magnitude est concentrée et peu
  informative, un simple indicateur binaire suffit.
- **Indicateurs `has_capital-gain` / `has_capital-loss`** (branche `num_bin`) :
  captent le signal **zéro / non-zéro**, qui porte l'essentiel de l'information.


In [ ]:
# capital-gain : brut vs log1p + RobustScaler
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(X_train["capital-gain"], ax=axes[0], bins=40, color="steelblue")
axes[0].set_title("capital-gain - brut")
axes[0].set_yscale("log")
sns.histplot(X_train_t["num_log__capital-gain"], ax=axes[1], bins=40, color="seagreen")
axes[1].set_title("capital-gain - log1p + RobustScaler")
fig.suptitle("capital-gain : log1p conserve la magnitude", fontsize=14)
fig.tight_layout()
plt.show()

# Indicateurs binaires has_* (capital-gain & capital-loss)
bin_cols = [f"num_bin__has_{c}" for c in BINARY_NUMERICAL_FEATURES]
prop = X_train_t[bin_cols].mean().rename("proportion à 1").round(4)
print("Part d'individus avec un capital non nul :")
print(prop.to_string())


### 4.4 Encodage one-hot des catégorielles (dont `workclass` → `Unknown`)

Chaque variable catégorielle de faible cardinalité est éclatée en une colonne
binaire par modalité. Deux branches one-hot :

- **`cat`** (marital-status, relationship, race, sex) : imputation par le **mode**
  (sans effet ici, ces colonnes n'ont aucun manquant) puis one-hot. **Toutes** les
  modalités sont conservées (pas de regroupement des rares).
- **`cat_unknown`** (`workclass`) : sa manquance est **informative**, donc les `?`
  deviennent une catégorie **`Unknown`** à part entière (colonne
  `workclass_Unknown`) au lieu d'être imputés par `Private`. Le signal est conservé.


In [ ]:
# Branches one-hot : cat (mode) + cat_unknown (workclass -> Unknown)
onehot_bases = MODE_CATEGORICAL_FEATURES + UNKNOWN_ONEHOT_FEATURES
onehot_cols = [c for c in feat_names if c.startswith("cat__") or c.startswith("cat_unknown__")]
print(f"{len(onehot_bases)} variables one-hot  ->  {len(onehot_cols)} colonnes\n")
for base in onehot_bases:
    n = sum(f"__{base}_" in c for c in feat_names)
    print(f"  {base:<16} -> {n:>3} modalites")

# Présence de la catégorie dédiée Unknown pour workclass
unknown_col = "cat_unknown__workclass_Unknown"
print(f"\n{unknown_col} : {int(X_train_t[unknown_col].sum())} individus "
      f"(= manquants workclass conservés, pas imputés)")


### 4.5 Variables à forte cardinalité : `occupation` et `native-country`

Deux traitements distincts, conformes au bilan §9 :

- **`occupation`** (`cat_occupation`) : trop de modalités pour le one-hot →
  `TargetEncoder`, qui remplace chaque modalité par une estimation (régularisée,
  cross-fittée donc **sans fuite**) du taux de `>50K`. La manquance étant
  informative, les `?` forment une catégorie **`Unknown`** dont le taux faible
  (~10 %) est capté par l'encodage.
- **`native-country`** (`cat_country`) : manquance **non** informative → imputation
  par le **mode** (US) ; ses ~20 modalités `<50` obs sont **regroupées**
  (`min_frequency = RARE_CATEGORY_MIN_FREQUENCY`) dans un bac `infrequent_sklearn`,
  puis one-hot.


In [ ]:
# occupation : valeur target-encodee vs taux reel de >50K par modalite
occ_col = "cat_occupation__occupation"
occ = pd.DataFrame({
    "occupation": X_train["occupation"].fillna("Unknown").values,
    "valeur_encodee": X_train_t[occ_col].values,
    "y": y_train_enc,
})
summary = (
    occ.groupby("occupation")
    .agg(taux_reel_50K=("y", "mean"), valeur_encodee=("valeur_encodee", "mean"), n=("y", "size"))
    .sort_values("taux_reel_50K", ascending=False)
    .round(3)
)
display(summary)

# native-country : effet du regroupement des modalites rares (<50 obs)
country_cols = [c for c in feat_names if c.startswith("cat_country__")]
n_raw = X_train["native-country"].nunique(dropna=True)
print(f"\nnative-country : {n_raw} modalites brutes  ->  {len(country_cols)} colonnes one-hot")
print("Bac des modalites rares present :",
      any("infrequent" in c for c in country_cols))


## 5. Comparaison des stratégies d'encodage : `onehot` vs `ordinal`

La pipeline accepte un paramètre `encoding`. Le one-hot convient aux modèles
linéaires mais gonfle la dimensionnalité ; l'ordinal est compact et adapté aux
modèles à base d'arbres.

In [ ]:
results = {}
for enc in ["onehot", "ordinal"]:
    p = PreprocessingPipeline(encoding=enc, scaling=True)
    p.fit_transform_target(y_train)
    pre_e = p.build()
    Xt = pre_e.fit_transform(X_train, y_train_enc)
    results[enc] = Xt.shape[1]

pd.DataFrame.from_dict(results, orient="index", columns=["n_colonnes_sortie"])


## 6. Transformation cohérente du jeu de test

Le test est transformé avec `transform` (et **non** `fit_transform`) : on
réutilise les statistiques apprises sur le train (médianes, moyennes/écarts,
modalités, encodage cible). Aucune fuite, et le test produit exactement les
mêmes colonnes.

In [ ]:
X_test_t = pre.transform(X_test)

print(f"Train transforme : {X_train_t.shape}")
print(f"Test  transforme : {X_test_t.shape}")
print("Memes colonnes           :", X_test_t.shape[1] == X_train_t.shape[1])
print("NaN dans le test         :", int(np.isnan(X_test_t).sum()))


## 7. Récapitulatif

- **`?` → NaN (au chargement) → traitement par variable** : plus aucun manquant en sortie.
- **Numériques normales** (age, education-num, hours-per-week) : recentrées/réduites (`StandardScaler`).
- **`capital-gain`** : `log1p` + `RobustScaler` (queue lourde conservée) **+** indicateur `has_capital-gain`.
- **`capital-loss`** : indicateur binaire `has_capital-loss` **seul** (pas de log).
- **Catégorielles faible cardinalité** (marital-status, relationship, race, sex) : one-hot, toutes modalités gardées.
- **`workclass`** : one-hot avec catégorie dédiée **`Unknown`** (manquance informative conservée).
- **`occupation`** : `TargetEncoder` (1 colonne suivant le taux de `>50K`) + catégorie `Unknown`.
- **`native-country`** : imputation mode (US) + regroupement des modalités `<50` obs, puis one-hot.
- **`education` / `fnlwgt`** : ignorées (`remainder="drop"`).
- La pipeline est **ajustée sur le train**, appliquée à l'identique au test → pas de data leakage.

Cette matrice transformée est l'entrée directe des modèles dans le notebook de
modélisation.
